In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                                DoubleType, DateType, TimestampType)

VOL = "/Volumes/workspace/default/maplebank"

customer_schema = StructType([
    StructField("customer_id",    StringType(), False),
    StructField("first_name",     StringType(), True),
    StructField("last_name",      StringType(), True),
    StructField("sin",            StringType(), True),   # PII — string, never number
    StructField("date_of_birth",  DateType(),   True),
    StructField("email",          StringType(), True),
    StructField("phone",          StringType(), True),
    StructField("street_address", StringType(), True),
    StructField("city",           StringType(), True),
    StructField("province",       StringType(), True),
    StructField("postal_code",    StringType(), True),
    StructField("customer_since", DateType(),   True),
])

account_schema = StructType([
    StructField("account_id",         StringType(), False),
    StructField("customer_id",        StringType(), False),
    StructField("account_number",     StringType(), True),  # String → keeps leading zeros!
    StructField("account_type",       StringType(), True),
    StructField("transit_number",     StringType(), True),
    StructField("institution_number", StringType(), True),
    StructField("open_date",          DateType(),   True),
    StructField("balance_cad",        DoubleType(), True),
    StructField("branch_id",          StringType(), True),
])

branch_schema = StructType([
    StructField("branch_id",      StringType(), False),
    StructField("branch_name",    StringType(), True),
    StructField("city",           StringType(), True),
    StructField("province",       StringType(), True),
    StructField("transit_number", StringType(), True),
])

txn_schema = StructType([
    StructField("transaction_id",        StringType(),    False),
    StructField("customer_id",           StringType(),    False),
    StructField("account_id",            StringType(),    False),
    StructField("branch_id",             StringType(),    True),
    StructField("transaction_date",      DateType(),      False),
    StructField("transaction_timestamp", TimestampType(), False),
    StructField("amount_cad",            DoubleType(),    False),
    StructField("transaction_type",      StringType(),    False),
    StructField("merchant_name",         StringType(),    True),
    StructField("channel",               StringType(),    True),
])

print("Schemas defined ✔")

Schemas defined ✔


In [0]:
df_customer = spark.read.option("header", True).schema(customer_schema).csv(f"{VOL}/dim_customer.csv")
df_account  = spark.read.option("header", True).schema(account_schema).csv(f"{VOL}/dim_account.csv")
df_branch   = spark.read.option("header", True).schema(branch_schema).csv(f"{VOL}/dim_branch.csv")
df_txn      = spark.read.option("header", True).schema(txn_schema).csv(f"{VOL}/fact_transactions.csv")

for name, df in [("customers", df_customer), ("accounts", df_account),
                 ("branches", df_branch), ("transactions", df_txn)]:
    print(f"{name:<14} {df.count():>7,} rows   {len(df.columns)} columns")

customers        1,000 rows   12 columns
accounts         2,025 rows   9 columns
branches            15 rows   5 columns
transactions    10,000 rows   10 columns


In [0]:
df_txn.printSchema()
df_txn.show(5, truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)
 |-- amount_cad: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- channel: string (nullable = true)

+--------------+-----------+----------+----------+----------------+---------------------+----------+----------------+-------------+-------+
|transaction_id|customer_id|account_id|branch_id |transaction_date|transaction_timestamp|amount_cad|transaction_type|merchant_name|channel|
+--------------+-----------+----------+----------+----------------+---------------------+----------+----------------+-------------+-------+
|TXN1000001    |CUST100231 |ACC500473 |BR_LAV_008|2025-11-24      |2025-11-24 17:13:00  |3625.93   |BILL_PAYMENT    |NULL     

In [0]:
def null_profile(df, name):
    print(f"\n── Null profile: {name} ──")
    df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).show(vertical=True)

null_profile(df_txn, "transactions")
null_profile(df_customer, "customers")


── Null profile: transactions ──
-RECORD 0---------------------
 transaction_id        | 0    
 customer_id           | 0    
 account_id            | 0    
 branch_id             | 0    
 transaction_date      | 0    
 transaction_timestamp | 0    
 amount_cad            | 0    
 transaction_type      | 0    
 merchant_name         | 7553 
 channel               | 0    


── Null profile: customers ──
-RECORD 0-------------
 customer_id    | 0   
 first_name     | 0   
 last_name      | 0   
 sin            | 0   
 date_of_birth  | 0   
 email          | 0   
 phone          | 0   
 street_address | 0   
 city           | 0   
 province       | 0   
 postal_code    | 0   
 customer_since | 0   



In [0]:
for name, df, key in [("transactions", df_txn, "transaction_id"),
                      ("customers", df_customer, "customer_id"),
                      ("accounts", df_account, "account_id")]:
    total = df.count()
    distinct = df.select(key).distinct().count()
    status = "✅ clean" if total == distinct else f"⚠ {total - distinct} duplicates!"
    print(f"{name:<14} total={total:>7,}  distinct {key}={distinct:>7,}  {status}")

transactions   total= 10,000  distinct transaction_id= 10,000  ✅ clean
customers      total=  1,000  distinct customer_id=  1,000  ✅ clean
accounts       total=  2,025  distinct account_id=  2,025  ✅ clean


In [0]:
# Transaction mix — what types flow through MapleBank?
df_txn.groupBy("transaction_type") \
      .agg(F.count("*").alias("count"), F.round(F.sum("amount_cad"), 2).alias("total_cad")) \
      .orderBy(F.col("total_cad").desc()) \
      .show()

# How many FINTRAC-territory transactions (≥ $10K)?
big = df_txn.filter(F.col("amount_cad") >= 10000).count()
print(f"Transactions ≥ $10,000 CAD: {big} ({big/df_txn.count()*100:.1f}%)")

+-----------------+-----+----------+
| transaction_type|count| total_cad|
+-----------------+-----+----------+
|   ATM_WITHDRAWAL| 1321|4763812.34|
|    INTERAC_DEBIT| 1232|4416708.25|
|             WIRE| 1235|4294626.64|
|INTERAC_ETRANSFER| 1240|4198230.09|
|     BILL_PAYMENT| 1260|4123145.69|
|  PAYROLL_DEPOSIT| 1265|4017394.74|
|              EFT| 1232|4005561.71|
|     POS_PURCHASE| 1215|3963899.24|
+-----------------+-----+----------+

Transactions ≥ $10,000 CAD: 303 (3.0%)
